In [121]:
import pandas as pd
import sqlite3
con = sqlite3.connect("C:\\Users\\fathi\\OneDrive\\Documents\\ojt semester3\\ipl-cricket-analytics\\data\\raw\\ipl.db")
def q(sql):
    return pd.read_sql(sql,con)

In [122]:
q('''
SELECT name
FROM sqlite_master
WHERE type IN ('table','view')
ORDER BY name;
''')

,name
0,deliveries
1,matches
2,matches_clean
3,players
4,teams
5,v_clean_matches
6,v_deliveries_clean
7,v_match_total
8,v_match_totals
9,v_matches_clean


using a CTE find how many distinct season are there
what is the highest number of matches played in any one season

data analysis= it is the process of inspecting, cleaning, and turning row confused numbers into clear useful facts

difference btw cte and view= a common table expression is a temperory, name query that exist only while your single sql statement runs, while a view is a same permanent virtual table stored in your database that you can reuse any time

rank()= it assigns a numerical position to each row in a dataset based on a specific order, cleaning gaps in the numbering if there are types

In [123]:
q('''
WITH per_season AS(
    SELECT DISTINCT season, COUNT(*) AS matches
    FROM matches
    GROUP BY season)

SELECT season,matches
FROM per_season
ORDER BY matches DESC
LIMIT 3;
''')

,season,matches
0,2013,76
1,2012,74
2,2022,74


In [124]:
q('''
WITH per_season AS(
    SELECT season,COUNT(*)AS matches
    FROM matches
    GROUP BY season)
SELECT
    COUNT(*) AS seasons,
    MAX(matches) AS busiest
FROM per_season;
''')

,seasons,busiest
0,19,76


using rank list the five highest run scores in ipl history with their rank member attach

In [125]:
q('''
WITH totals AS (
    SELECT batter, SUM(batter_runs) AS runs
    FROM deliveries
    GROUP BY batter
)
SELECT batter,runs,
       RANK() OVER (ORDER BY runs DESC) AS rk
FROM totals
ORDER BY rk
LIMIT 5;
''')

,batter,runs,rk
0,V Kohli,9050,1
1,RG Sharma,7185,2
2,S Dhawan,6769,3
3,DA Warner,6567,4
4,KL Rahul,5668,5


**1st innings**= the team that wins toss get their turn to bat for maximum of 20 overs to set a target score

**2nd innings**= opposing team then gets their turn to that for 20 overs to chase down that target score

In [126]:
q('''
SELECT COUNT(*) AS before_join
FROM matches_clean;
''')

,before_join
0,1212


In [127]:
q("""
WITH ranked_venues AS (
    SELECT
        *,
        ROw_NUMBER() OVER(
            PARTITION BY venue
            ORDER BY venue_id
            ) AS rn
        FROM venues
),
dedup_venues AS(
    SELECT *
    FROM ranked_venues
    WHERE rn = 1
)
SELECT COUNT(*)AS after_join
FROM matches_clean m
LEFT JOIN dedup_venues v
    ON m.venue = v.venue;
""")

,after_join
0,1212


In [128]:
q('''
WITH first_innings AS(
    SELECT
        match_id,
        SUM(total_runs) AS first_innings_total
    FROM deliveries
    WHERE innings = 1
    GROUP BY match_id
)
SELECT *
FROM first_innings
LIMIT 10;
''')

,match_id,first_innings_total
0,335982,222
1,335983,240
2,335984,129
3,335985,165
4,335986,110
5,335987,166
6,335988,142
7,335989,208
8,335990,214
9,335991,182


In [129]:
q('''
SELECT
    match_id,
    innings,
    SUM(total_runs) AS check_total
FROM deliveries
WHERE match_id =335982
    AND innings = 1
GROUP BY match_id,innings;
''')

,match_id,innings,check_total
0,335982,1,222


In [130]:
q('''
SELECT
    match_id,
    innings,
    SUM(total_runs) AS check_total
FROM deliveries
WHERE match_id IN (335982,335983,335984,335985,335986)
    AND innings = 1
GROUP BY match_id
ORDER BY match_id
''')

,match_id,innings,check_total
0,335982,1,222
1,335983,1,240
2,335984,1,129
3,335985,1,165
4,335986,1,110


In [131]:
q('''
SELECT match_id, 
       COUNT(DISTINCT batting_team) AS batting_teams
FROM deliveries
WHERE innings= 1
GROUP BY match_id
HAVING COUNT(DISTINCT batting_team) != 1;
''')

,match_id,batting_teams


In [132]:
q('''
WITH first_innings AS (
    SELECT
        match_id,
        batting_team AS innings1_batting_team,
        SUM(total_runs) AS first_innings_total
    FROM deliveries
    WHERE innings = 1
    GROUP BY match_id, batting_team
)
SELECT *
FROM first_innings
LIMIT 10;
''')

,match_id,innings1_batting_team,first_innings_total
0,335982,Kolkata Knight Riders,222
1,335983,Chennai Super Kings,240
2,335984,Rajasthan Royals,129
3,335985,Mumbai Indians,165
4,335986,Sunrisers Hyderabad,110
5,335987,Punjab Kings,166
6,335988,Sunrisers Hyderabad,142
7,335989,Chennai Super Kings,208
8,335990,Sunrisers Hyderabad,214
9,335991,Punjab Kings,182


In [133]:
q("""
WITH first_innings AS (
    SELECT
        match_id,
        batting_team AS innings1_batting_team,
        SUM(total_runs) AS first_innings_total
    FROM deliveries
    WHERE innings = 1
    GROUP BY match_id, batting_team
)
SELECT m.match_id, m.season_year,f.innings1_batting_team,f.first_innings_total
FROM matches_clean m
LEFT JOIN first_innings f
    ON m.match_id = f.match_id
LIMIT 5;
""")

,match_id,season_year,innings1_batting_team,first_innings_total
0,335982,2008,Kolkata Knight Riders,222
1,1082591,2017,Sunrisers Hyderabad,207
2,1082592,2017,Mumbai Indians,184
3,1082593,2017,Gujarat Lions,183
4,1082594,2017,Rising Pune Supergiant,163


In [134]:
con.executescript("""
DROP VIEW IF EXISTS v_match_totals;

CREATE VIEW v_match_totals AS
WITH first_innings AS (
    SELECT 
        match_id,
        batting_team AS innings1_batting_team,
        SUM(total_runs) AS first_innings_total
    FROM deliveries
    WHERE innings = 1
    GROUP BY match_id, batting_team
)
SELECT 
    m.*,
    f.innings1_batting_team,
    f.first_innings_total
FROM matches_clean m
LEFT JOIN first_innings f 
    ON m.match_id = f.match_id
;
""")

con.commit()

In [135]:
q("""
SELECT COUNT(*) AS total_rows
FROM v_match_totals;
""")

,total_rows
0,1212


**LAG**= it is a window function that lets view look back at data from a previous row

1. LAG()

Season          Matches
2021              60
2022              74
2023              74

2. LAG(matches)

Season          Matches         Previous
2021              60              NULL
2022              74               60
2023              74               74

3. Season-on-Season Change

2021–2022: 74 − 60 = 14

2022–2023: 74 − 74 = 0

In [136]:
q("""
SELECT season_year, COUNT(DISTINCT match_id) AS match_count
FROM matches_clean
GROUP BY season_year
ORDER BY season_year ASC;
""")

,season_year,match_count
0,2008,58
1,2009,57
2,2010,60
3,2011,73
4,2012,74
5,2013,76
6,2014,60
7,2015,59
8,2016,60
9,2017,59


In [137]:
q("""
WITH season_counts AS (
    SELECT
        season_year,
        COUNT(*) AS matches
    FROM matches_clean
    GROUP BY season_year
)
SELECT
    season_year,
    matches,
    LAG(matches) OVER (
        ORDER BY season_year
    ) AS previous_season_matches
FROM season_counts
ORDER BY season_year;
""")

,season_year,matches,previous_season_matches
0,2008,58,NaN
1,2009,57,58.0
2,2010,60,57.0
3,2011,73,60.0
4,2012,74,73.0
5,2013,76,74.0
6,2014,60,76.0
7,2015,59,60.0
8,2016,60,59.0
9,2017,59,60.0


In [138]:
q("""
WITH season_counts AS (
    SELECT
        season_year,
        COUNT(*) AS matches
    FROM matches_clean
    GROUP BY season_year
),
with_previous AS (
    SELECT
        season_year,
        matches,
        LAG(matches) OVER (
            ORDER BY season_year
        ) AS previous_season_matches
    FROM season_counts
)
SELECT
    season_year,
    matches,
    previous_season_matches,
    matches - previous_season_matches AS season_change
FROM with_previous
ORDER BY season_year;
""")

,season_year,matches,previous_season_matches,season_change
0,2008,58,NaN,NaN
1,2009,57,58.0,-1.0
2,2010,60,57.0,3.0
3,2011,73,60.0,13.0
4,2012,74,73.0,1.0
5,2013,76,74.0,2.0
6,2014,60,76.0,-16.0
7,2015,59,60.0,-1.0
8,2016,60,59.0,1.0
9,2017,59,60.0,-1.0


**power play**= A specific set of overs at start of an innings where the feilding team must keep most of their players close to the center,creating big gaps in the outfield to help batters score more runs.

**death overs**= final few overs, batters try to score as many runs as possible by hitting boundaries and 6s, bowlers try to stop runs and take wickets using slow balls.

**chasing v/s defending**= chasing means batting seconds to score more runs  than the target set by the first team, while defending teams bowling and feilding second to stop the batting team from reaching that target

In [139]:
q('''
SELECT
    MIN(over_number) AS min_over,
    MAX(over_number) AS max_over,
    COUNT(DISTINCT over_number) AS distinct_overs
FROM deliveries
WHERE is_super_over = 0;
''')

,min_over,max_over,distinct_overs
0,0,19,20


0-9 = overs

0-5 = power play

6-14 = middle overs

15-19 = death overs

In [140]:
q('''
SELECT
    ROUND(
    6.0 * SUM(total_runs)/ COUNT(*),
    2
    ) AS powerplay_rpo
FROM deliveries
WHERE is_super_over = 0
    AND is_wide_ball = 0
    AND is_no_ball = 0
    AND over_number BETWEEN 0 AND 5;
''')

,powerplay_rpo
0,7.71


In [141]:
q('''
SELECT
    ROUND(
        6.0 * SUM(total_runs) / COUNT(*),
        2
    ) AS death_over_rpo
FROM deliveries
WHERE is_super_over = 0
    AND is_wide_ball = 0
    AND is_no_ball = 0
    AND over_number BETWEEN 15 AND 19;
''')

,death_over_rpo
0,9.61


In [142]:
q('''
SELECT
    ROUND(
        100.0 * SUM(is_wicket) / COUNT(*),
        2
    ) AS powerplay_wicket_rate
FROM deliveries
WHERE is_super_over = 0
    AND is_wide_ball = 0
    AND is_no_ball = 0
    AND over_number BETWEEN 0 AND 5;
''')

,powerplay_wicket_rate
0,4.0


In [143]:
q('''
SELECT
    ROUND(
        100.0 * SUM(is_wicket) / COUNT(*),
        2
    ) AS deathovers_wicket_rpo
FROM deliveries
WHERE is_super_over = 0
    AND is_wide_ball = 0
    AND is_no_ball = 0
    AND over_number BETWEEN 15 AND 19;
''')

,deathovers_wicket_rpo
0,8.41


deathovers are harder to ball because scoring is higher and wickets also fall much more frequently.

In [144]:
q('''
SELECT
    ROUND(
        6.0 * SUM(total_runs) / COUNT(*),
        2
    ) AS middleover_rpo
FROM deliveries
WHERE is_super_over = 0
    AND is_wide_ball = 0
    AND is_no_ball = 0
    AND over_number BETWEEN 6 AND 14;
''')

,middleover_rpo
0,7.67


dotball= a delivery in which the batters scores 0 runs

In [145]:
q('''
SELECT ROUND(
        100.0 * SUM(CASE WHEN batter_runs = 0 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
) AS batter_dot
FROM deliveries
WHERE is_super_over = 0
    AND is_wide_ball = 0
    AND is_no_ball = 0;
''')

,batter_dot
0,37.45


In [146]:
q('''
SELECT ROUND(
        100.0 * SUM(CASE WHEN total_runs = 0 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
) AS batter_dot
FROM deliveries
WHERE is_super_over = 0
    AND is_wide_ball = 0
    AND is_no_ball = 0;
''')

,batter_dot
0,35.64


analysis

1- number/finding- Wankhede Stadium	130 matches

2- what exactly was counted- number of matches played at Wankhede Stadium using the clean venue field

3- population it came from- All matches in matches_clean

4- what decision would this finding change- Helps ensure the venue level analysis and comparisons are based on cleaned venue names rather than fragemented row names.

In [147]:
q('''
SELECT venue_clean,COUNT(*) AS matches
FROM matches_clean
GROUP BY venue_clean
ORDER BY matches DESC
LIMIT 10;
''')

,venue_clean,matches
0,Wankhede Stadium,130
1,M Chinnaswamy Stadium,104
2,Eden Gardens,104
3,MA Chidambaram Stadium,95
4,Rajiv Gandhi International Stadium,87
5,Sawai Mansingh Stadium,66
6,Feroz Shah Kotla,60
7,Dubai International Cricket Stadium,46
8,Arun Jaitley Stadium,41
9,Narendra Modi Stadium,37


descive match- a match that entered with a clear winner

In [148]:
q('''
SELECT
    COUNT(*) AS decisive_matches,
    SUM(
        CASE
            WHEN toss_winner = match_winner THEN 1
            ELSE 0
        END
        ) AS toss_winner_won,
        ROUND(
        100.0 * SUM(
            CASE
                WHEN toss_winner = match_winner THEN 1
                ELSE 0
            END
            )/COUNT(*),
            2
            ) AS percentage
FROM matches_clean
WHERE result ='win';
''')

,decisive_matches,toss_winner_won,percentage
0,1187,613,51.64


1- 613 of 1187 (51.64%)

2- matches where toss_winner = match_winner

3-1187 decisive matches where result = win

4-Helps access whether winning the toss is accossiated with a meaningfull match winning adavantage.

In [149]:
q('''
SELECT 
    season_year,
    COUNT(*) AS total_matches,
    SUM(CASE WHEN toss_decision ='field' THEN 1 ELSE 0 END) AS field_first,
    ROUND(
        100.0 * SUM(CASE WHEN toss_decision ='field' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
        ) AS field_first_pct
FROM matches_clean
GROUP BY season_year
ORDER BY season_year;
''')

,season_year,total_matches,field_first,field_first_pct
0,2008,58,32,55.17
1,2009,57,22,38.60
2,2010,60,21,35.00
3,2011,73,48,65.75
4,2012,74,37,50.00
5,2013,76,31,40.79
6,2014,60,41,68.33
7,2015,59,34,57.63
8,2016,60,49,81.67
9,2017,59,48,81.36


1- finding/number = 55.17 in 2008, 38.6% in 2009,35% in 2010, above 80 2016 to 2019

2- percentage of matches in each season where toss winning team choose the field

3- all matches in each season in matches_clean

4-helps identify how toss strategy and preference for fielding first changed over time.

caught dismissals= happens when the batter hits the goal with their bat or gloved hand holding the bat,and a feilder or the bowler catches the ball in the air before it touchess the ground

In [150]:
q('''
SELECT
    COUNT(*) AS caught_dismissals
FROM deliveries
WHERE wicket_kind = 'caught';
''')

,caught_dismissals
0,9056


1-caught dismissals = 9056

2-delivery records there wicket_kind= caught

3- dismissals records in the delivery table

4- helps identify how important caught dismissals are as mode of taking wickets

chasing win rate is the percentage of matches won by the team batting second.

In [152]:
q('''
SELECT COUNT(*) AS decisive_matches
FROM v_match_totals
WHERE result ='win'
    AND first_innings_total IS NOT NULL;
''')

,decisive_matches
0,1187


In [ ]:
q('''
SELECT
    CASE
        WHEN first_innings_total < 140 THEN 'Under 140'
        WHEN first_innings_total BETWEEN 140 AND 159 THEN '140-159'
        WHEN first_innings_total BETWEEN 160 AND 179 THEN '160-179'
        WHEN first_innings_total BETWEEN 180 AND 199 THEN '180-199'
        ELSE '200+'
    END AS score_band,
    COUNT(*) AS matches
FROM v_match_totals
WHERE result = 'win'
    AND first_innings_total IS NOT NULL
GROUP BY score_band;
''')

,score_band,matches
0,140-159,241
1,160-179,312
2,180-199,217
3,200+,198
4,Under 140,219


In [154]:
q('''
SELECT
    match_id,
    first_innings_total,
    innings1_batting_team,
    match_winner,
    CASE
        WHEN match_winner != innings1_batting_team THEN 1
        ELSE 0
    END AS chase_win
FROM v_match_totals
WHERE result ='win'
    AND first_innings_total IS NOT NULL
LIMIT 10;
''')

,match_id,first_innings_total,innings1_batting_team,match_winner,chase_win
0,335982,222,Kolkata Knight Riders,Kolkata Knight Riders,0
1,1082591,207,Sunrisers Hyderabad,Sunrisers Hyderabad,0
2,1082592,184,Mumbai Indians,Rising Pune Supergiant,1
3,1082593,183,Gujarat Lions,Kolkata Knight Riders,1
4,1082594,163,Rising Pune Supergiant,Punjab Kings,1
5,1082595,157,Royal Challengers Bangalore,Royal Challengers Bangalore,0
6,1082596,135,Gujarat Lions,Sunrisers Hyderabad,1
7,1082597,178,Kolkata Knight Riders,Mumbai Indians,1
8,1082598,148,Royal Challengers Bangalore,Punjab Kings,1
9,1082599,205,Delhi Capitals,Delhi Capitals,0


In [155]:
q('''
WITH chase_data AS (
    SELECT
        first_innings_total,
        
        CASE
            WHEN first_innings_total < 140 THEN 'Under 140'
            WHEN first_innings_total BETWEEN 140 AND 159 THEN '140-159'
            WHEN first_innings_total BETWEEN 160 AND 179 THEN '160-179'
            WHEN first_innings_total BETWEEN 180 AND 199 THEN '180-199'
            ELSE '200+'
        END AS score_band,
        CASE
            WHEN match_winner != innings1_batting_team THEN 1
        ELSE 0
    END AS chase_win
FROM v_match_totals
WHERE result ='win'
    AND first_innings_total IS NOT NULL
)

SELECT
    score_band,
    COUNT(*) AS sample_size,
    ROUND(
        100.0*SUM(chase_win)/COUNT(*),
        1
        ) AS chase_win_rate
FROM chase_data
GROUP BY score_band
ORDER BY 
    CASE score_band
        WHEN 'Under 140' THEN 1
        WHEN '140-159' THEN 2
        WHEN '169-179' THEN 3
        WHEN '180-199' THEN 4
        WHEN '200+' THEN 5
    END;
''')

,score_band,sample_size,chase_win_rate
0,160-179,312,53.2
1,Under 140,219,82.6
2,140-159,241,71.0
3,180-199,217,38.2
4,200+,198,23.2


higher first innings totals correspond to lower chase win rates

In [162]:
q('''
SELECT
    COUNT(*) AS matches,
    SUM(
        CASE
            WHEN win_by_wickets IS NOT NULL THEN 1
            ELSE 0
        END
    ) AS chase_wins,
    ROUND(100.0 * SUM(CASE WHEN win_by_wickets IS NOT NULL THEN 1 ELSE 0 END)
      / COUNT(*),
      1)AS chase_win_rate
FROM v_match_totals
WHERE result ='win'
    AND first_innings_total < 140;
''')

,matches,chase_wins,chase_win_rate
0,219,177,80.8


In [158]:
q("""
WITH chase_defs AS (
    SELECT
        match_id,

        CASE
            WHEN match_winner != innings1_batting_team THEN 1
            ELSE 0
        END AS definition_1,

        CASE
            WHEN win_by_wickets IS NOT NULL THEN 1
            ELSE 0
        END AS definition_2

    FROM v_match_totals
    WHERE result = 'win'
      AND first_innings_total IS NOT NULL
)

SELECT
    COUNT(*) AS total_matches,
    SUM(
        CASE
            WHEN definition_1 = definition_2 THEN 1
            ELSE 0
        END
    ) AS agreements,
    SUM(
        CASE
            WHEN definition_1 != definition_2 THEN 1
            ELSE 0
        END
    ) AS disagreements,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN definition_1 = definition_2 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        1
    ) AS agreement_pct
FROM chase_defs;
""")

,total_matches,agreements,disagreements,agreement_pct
0,1187,1181,6,99.5


the two chase definitions agree on 1181 out of 1187 decing matches or 99.5% they disagree on only 6 matches but those six are enough to slightly change the reported chase win rate.

In [163]:
chase_bands = q("""
WITH chase_data AS (
    SELECT
        first_innings_total,

        CASE
            WHEN first_innings_total < 140 THEN 'Under 140'
            WHEN first_innings_total BETWEEN 140 AND 159 THEN '140-159'
            WHEN first_innings_total BETWEEN 160 AND 179 THEN '160-179'
            WHEN first_innings_total BETWEEN 180 AND 199 THEN '180-199'
            ELSE '200+'
        END AS score_band,

        CASE
            WHEN match_winner != innings1_batting_team THEN 1
            ELSE 0
        END AS chase_win

    FROM v_match_totals
    WHERE result = 'win'
      AND first_innings_total IS NOT NULL
)

SELECT
    score_band,
    COUNT(*) AS sample_size,
    ROUND(
        100.0 * SUM(chase_win) / COUNT(*),
        1
    ) AS chase_win_rate
FROM chase_data
GROUP BY score_band
ORDER BY
    CASE score_band
        WHEN 'Under 140' THEN 1
        WHEN '140-159' THEN 2
        WHEN '160-179' THEN 3
        WHEN '180-199' THEN 4
        WHEN '200+' THEN 5
    END;
""")

chase_bands

,score_band,sample_size,chase_win_rate
0,Under 140,219,82.6
1,140-159,241,71.0
2,160-179,312,53.2
3,180-199,217,38.2
4,200+,198,23.2


matplot library - it is a popular open sourse python library use for creating static,interactive and animated data visualizations

different charts and their uses- line chart= to show trends or changes over time(plt.plot(x,y))
bar chart= to compare quantity across different categories(plt.bar(x,y))
scatter plot= to observe relationship,pattern,or core relation btw to variables(plt.scatter(x,y))
Histogram = to show the frequency distribution of data(plt.his(x))
pie chart= display propotion,percentages or part to whole data (plt.pie(x))
box plot = to easily see the spread,variability,and outliers with in a dataset(plt.boxplot(x))